In [1]:
import subprocess

result = subprocess.run(
    ["python", "-m", "src.retinocare.models.train",
     "--config", "../configs/train_config.yaml",
     "--model", "baseline_cnn"],
    cwd="..", capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


STDERR: Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Python\retinocare-ai\src\retinocare\models\train.py", line 145, in <module>
    main()
  File "d:\Python\retinocare-ai\src\retinocare\models\train.py", line 79, in main
    with open(args.config) as f:
         ^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '../configs/train_config.yaml'



In [ ]:
import torch, yaml
from src.retinocare.data.dataset import get_dataloaders
from src.retinocare.models.baseline_cnn import BaselineCNN
from src.retinocare.evaluation.metrics import compute_classification_report, plot_confusion_matrix

with open("../configs/train_config.yaml") as f:
    config = yaml.safe_load(f)

_, _, test_dl = get_dataloaders(config)

model = BaselineCNN(num_classes=5)
model.load_state_dict(torch.load("../models/baseline_cnn.pt", map_location="cpu"))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_dl:
        logits = model(images)
        all_preds.extend(logits.argmax(dim=1).tolist())
        all_labels.extend(labels.tolist())

SEVERITY_LABELS = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
report = compute_classification_report(all_labels, all_preds, SEVERITY_LABELS)
print("Weighted F1:", report["weighted avg"]["f1-score"])
print("Macro F1:", report["macro avg"]["f1-score"])

plot_confusion_matrix(all_labels, all_preds, SEVERITY_LABELS, "../docs/baseline_cnn_confusion_matrix.png")
print("Confusion matrix saved to docs/baseline_cnn_confusion_matrix.png")